# Simload CSV comparison

This notebook calls functions from `analyze_simload_csv.py`.

Assumption: `analyze_simload_csv.py` is in the same directory as this notebook, or otherwise available on `PYTHONPATH`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from analyze_simload_csv import (
    read_many_csv,
    infer_column,
    infer_metric_columns,
    save_summary,
    plot_timeseries,
    plot_boxplot,
    plot_completion_throughput,
    TIME_CANDIDATES,
    EVENT_ID_CANDIDATES,
)

## Configure input files

Change `csv_glob` to point to your Simload CSV outputs.

In [ ]:
csv_glob = "simload_runs/*.csv"
outdir = Path("simload_analysis")
outdir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(Path(".").glob(csv_glob))
csv_files

## Load and combine CSV files

In [ ]:
df = read_many_csv(csv_files)
df.to_csv(outdir / "combined.csv", index=False)

print(f"rows: {len(df)}")
print(f"runs: {df['run'].nunique()}")
df.head()

## Detect time and metric columns

You can override these manually if auto-detection does not match your CSV schema.

In [ ]:
time_col = infer_column(df.columns, TIME_CANDIDATES)
metrics = infer_metric_columns(df)

print("time column:", time_col)
print("metrics:")
for metric in metrics:
    print("  ", metric)

Optional manual override:

In [ ]:
# Example manual overrides; uncomment/edit as needed.
# time_col = "end_offset_s"
# metrics = [
#     "event_latency_s",
#     "cpu_fraction",
#     "gpu_fraction",
#     "cpu_work_s",
#     "gpu_work_s",
#     "gpu_memory_mb",
# ]

## Summary table by run

In [ ]:
summary = save_summary(df, metrics, outdir)
summary

## Generate comparison plots

The plot functions save PNG files into `outdir`.

In [ ]:
if time_col is not None:
    for metric in metrics:
        if metric != time_col:
            plot_timeseries(df, time_col, metric, outdir)

for metric in metrics:
    plot_boxplot(df, metric, outdir)

id_col = infer_column(df.columns, EVENT_ID_CANDIDATES)
if time_col is not None and id_col is not None:
    plot_completion_throughput(
        df=df,
        time_col=time_col,
        id_col=id_col,
        outdir=outdir,
        bin_width_s=1.0,
    )

sorted(outdir.glob("*.png"))

## Display generated plots

In [ ]:
from IPython.display import Image, display

for path in sorted(outdir.glob("*.png")):
    print(path)
    display(Image(filename=str(path)))